# Chapter 1: XGBoost — The Industry-Standard Default

In [ ]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

X = pd.get_dummies(df.drop(columns=["churn"]), drop_first=True)     # XGBoost needs numeric input -> one-hot first
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

xgb_model = XGBClassifier(
    n_estimators=200,        # how many trees to build, sequentially
    max_depth=4,             # how deep each individual tree can grow
    learning_rate=0.05,      # how much each new tree corrects the previous ensemble (a "shrinkage" factor)
    subsample=0.8,           # use 80% of ROWS for each tree — adds randomness, reduces overfitting
    colsample_bytree=0.8,    # use 80% of COLUMNS for each tree — same purpose
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1                 # use every available CPU core
)
xgb_model.fit(X_train, y_train)

pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost ROC-AUC:", round(roc_auc_score(y_test, pred_xgb), 4))

The five hyperparameters shown here are the ones worth understanding deeply, because they reappear (often under near-identical names) in LightGBM and CatBoost too:

- **`n_estimators`** — more trees generally improve performance, up to a point of diminishing returns and eventual overfitting. Think of it as "how long you let the model keep correcting its own mistakes."
- **`max_depth`** — deeper trees capture more complex interactions between features but overfit more easily. 3–6 is a sane starting range for most business tabular problems.
- **`learning_rate`** — smaller values (0.01–0.1) make each tree's correction more conservative, generally producing better final performance at the cost of needing more trees (`n_estimators`) to compensate. This tradeoff between `learning_rate` and `n_estimators` is the single most important pair of knobs to jointly consider when tuning any boosting model.
- **`subsample` / `colsample_bytree`** — both inject randomness by training each tree on a random subset of rows/columns rather than the full data. This is directly analogous to a Random Forest's bootstrap sampling, and it's specifically a defense against overfitting — without it, boosting's aggressive error-correction can start memorizing noise in the training data.
- **`n_jobs=-1`** — a small but important practical setting: use every CPU core available for training, since tree building parallelizes well.

💡 `n_jobs=-1` is worth remembering as a general scikit-learn-ecosystem convention — it appears across dozens of functions (cross_val_score, GridSearchCV, RandomForestClassifier, and all three boosting libraries) and always means "use all available cores."

### Cheat Sheet — XGBoost

| Task | Code |
|---|---|
| Classifier | `XGBClassifier(n_estimators, max_depth, learning_rate, subsample, colsample_bytree)` |
| Regressor | `XGBRegressor(...)` — identical hyperparameters, for continuous targets |
| Feature importance | `model.feature_importances_` |
| Early stopping | `model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=20)` |

---